# Music Annotation Project V8.0b: Raw Spatial Topology Controlled Challenger

**Runtime:** GPU. Select **Runtime > Run all**.

This experiment tests one narrow question: **does raw 33×33 spatial topology improve the safe coverage frontier over a rerun of the V8.0a scalar logistic baseline?**

The two arms share the same frozen V7.5 development roster, label construction, sampled centres, source-group folds, calibration method, decision thresholds, safety metrics and frozen V7.8 comparator. The intended experimental difference is the observable input representation and the minimum estimator required to consume it.

This notebook is development-only. It does not access frozen V7.8 independent outcomes, does not authorise production promotion, and fails closed without `development_score_manifest.csv`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn pyarrow torch torchvision matplotlib


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, sys, platform
import cv2, joblib, torch
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.isotonic import IsotonicRegression

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
MANIFEST_PATH=ROOT/'development_score_manifest.csv'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_0b'
V75_RUN_OVERRIDE=None
SEED=42; PATCH=33; HALF=PATCH//2; DARK_THRESHOLD=200; TRUTH_TOLERANCE=1
MAX_PER_PAGE_CLASS=25000; BATCH=512; EPOCHS=12; LR=1e-3; WEIGHT_DECAY=1e-4
PRINT_THRESHOLD=.995; ANNOTATION_THRESHOLD=.995; PRINT_MARGIN=.990
THRESHOLD_GRID=np.unique(np.r_[np.linspace(.90,.99,10),np.linspace(.991,.999,9),.9995,.9999])
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('Select a GPU runtime.')
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_0b_topology_controlled';OUT=OUTPUT_ROOT/RUN
TABLES=OUT/'tables';MODELS=OUT/'models';AUDIT=OUT/'audit';CHARTS=OUT/'charts'
for p in [OUT,TABLES,MODELS,AUDIT,CHARTS]:p.mkdir(parents=True,exist_ok=False)
print('Device:',DEVICE,'Output:',OUT)


## 1. Authoritative development roster and source-group gate

The manifest must contain `page`, `source_group`, `score_title`, `edition_or_source`, and `acquisition_notes`. Page is never used as a model input. Source group is used only for fold separation and reporting.


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def latest(root,required):
 h=[]
 if root.exists():
  for p in root.rglob('*'):
   if p.is_dir() and all((p/r).exists() for r in required):h.append(p)
 return max(h,key=lambda p:p.stat().st_mtime) if h else None
def imap(folder):return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def objhash(v):return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest(V75_ROOT,['tables/cache_status.csv'])
if V75 is None:raise RuntimeError('Complete V7.5 development run not found.')
if not MANIFEST_PATH.exists():raise RuntimeError(f'Missing authoritative source-group manifest: {MANIFEST_PATH}')
manifest=pd.read_csv(MANIFEST_PATH,dtype=str)
required={'page','source_group','score_title','edition_or_source','acquisition_notes'}
if required-set(manifest):raise RuntimeError(f'Manifest missing columns: {sorted(required-set(manifest))}')
if manifest.page.duplicated().any() or manifest.source_group.isna().any() or (manifest.source_group.str.strip()=='').any():raise RuntimeError('Manifest page values must be unique and source_group complete.')
status=pd.read_csv(V75/'tables/cache_status.csv');cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
clean_map=imap(ROOT/'source'/'clean');obs_map=imap(ROOT/'aligned');pages=sorted(set(cache)&set(clean_map)&set(obs_map))
if set(manifest.page)!=set(pages):
 raise RuntimeError(f'Manifest/V7.5 page mismatch. Missing={sorted(set(pages)-set(manifest.page))[:20]} Extra={sorted(set(manifest.page)-set(pages))[:20]}')
meta=manifest.set_index('page');print('Pages:',len(pages),'Source groups:',manifest.source_group.nunique())


## 2. Load frozen arrays and construct identical truth labels

Both arms use the same eligible pixels and truth map. Clean truth is retained only in this label/scoring stage.


In [ ]:
def printed_ink(clean):return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def truth_maps(clean):
 ink=printed_ink(clean);k=np.ones((2*TRUTH_TOLERANCE+1,2*TRUTH_TOLERANCE+1),np.uint8);near=cv2.dilate(ink.astype(np.uint8),k)>0;er=cv2.erode(ink.astype(np.uint8),k)>0;boundary=np.logical_xor(near,er);return ink,near,boundary
def background_surface(obs,mask,sigma=45.):
 vis=(~mask)&(obs>=200);num=cv2.GaussianBlur(obs.astype(np.float32)*vis,(0,0),sigma);den=cv2.GaussianBlur(vis.astype(np.float32),(0,0),sigma);fallback=cv2.GaussianBlur(obs,(0,0),sigma);return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)
runtime={};load_rows=[]
for page in pages:
 with np.load(cache[page]) as z:
  req=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact'];miss=[k for k in req if k not in z.files]
  if miss:raise RuntimeError(f'{page}: missing {miss}')
  arr={k:z[k] for k in req}
 clean=cv2.imread(str(clean_map[page]),0);obs=cv2.imread(str(obs_map[page]),0)
 mask=arr['mask']>0
 if clean is None or obs is None or clean.shape!=obs.shape or any(v.shape!=obs.shape for v in arr.values()):raise RuntimeError(f'{page}: shape/read failure')
 if not np.array_equal(arr['geometry_t80'][~mask],obs[~mask]):raise RuntimeError(f'{page}: V7.8 outside-mask drift')
 ink,near,boundary=truth_maps(clean);eligible=mask&(obs<DARK_THRESHOLD)&~(arr['g80_preserve']>0);label=np.full(obs.shape,-1,np.int8);label[eligible&near&~boundary]=1;label[eligible&~near&~boundary]=0;label[eligible&boundary]=2
 paper=background_surface(obs,mask);norm=cv2.divide(obs,np.maximum(paper,1),scale=255);edge=cv2.Canny(obs,70,170)>0;depth=cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5)
 runtime[page]={'clean':clean,'obs':obs,'mask':mask,'baseline':arr['geometry_t80'],'label':label,'norm':norm,'edge':edge,'depth':depth,
  'preserve':arr['g80_preserve']>0,'remove':arr['g80_remove']>0,'ambiguous':arr['g80_ambiguous']>0,'horizontal':arr['g80_horizontal']>0,'vertical':arr['g80_vertical']>0,'compact':arr['g80_compact']>0}
 load_rows.append({'page':page,'source_group':meta.loc[page,'source_group'],'eligible':int(eligible.sum()),'print':int((label==1).sum()),'annotation':int((label==0).sum()),'uncertain_truth':int((label==2).sum()),'cache_sha256':sha256_file(cache[page])})
pd.DataFrame(load_rows).to_csv(AUDIT/'label_and_cache_audit.csv',index=False)


## 3. One paired centre sample and one deterministic source-group fold map

The same centre rows are passed to both arms. The split audit proves no source group crosses a fold boundary.


In [ ]:
rng=np.random.default_rng(SEED); centres=[]
for page,z in runtime.items():
 for cls in [0,1]:
  ys,xs=np.where(z['label']==cls);idx=np.arange(len(ys));take=idx if len(idx)<=MAX_PER_PAGE_CLASS else rng.choice(idx,MAX_PER_PAGE_CLASS,replace=False)
  for j in take:centres.append({'page':page,'source_group':str(meta.loc[page,'source_group']),'y':int(ys[j]),'x':int(xs[j]),'truth':cls})
centres=pd.DataFrame(centres).sort_values(['source_group','page','y','x']).reset_index(drop=True);centres['sample_id']=np.arange(len(centres));centres.to_csv(TABLES/'paired_sample_manifest.csv',index=False)
groups=centres.source_group;ng=groups.nunique();splitter=LeaveOneGroupOut() if ng<=10 else GroupKFold(n_splits=min(10,ng))
fold=np.full(len(centres),-1,int);audit=[]
for f,(tr,te) in enumerate(splitter.split(centres,centres.truth,groups),1):
 fold[te]=f;train=set(groups.iloc[tr]);test=set(groups.iloc[te]);over=train&test
 audit.append({'fold':f,'train_groups':len(train),'test_groups':len(test),'overlap_groups':len(over),'test_rows':len(te),'test_pages':centres.iloc[te].page.nunique()})
 if over:raise RuntimeError(f'Fold {f}: source-group leakage {over}')
centres['fold']=fold;centres.to_csv(TABLES/'paired_sample_manifest.csv',index=False);pd.DataFrame(audit).to_csv(AUDIT/'source_group_split_audit.csv',index=False)
fold_payload=centres[['sample_id','page','source_group','fold']].to_dict('records');(AUDIT/'fold_map.json').write_text(json.dumps({'sha256':objhash(fold_payload),'rows':fold_payload},indent=2));print(centres.truth.value_counts(), 'folds',sorted(centres.fold.unique()))


## 4. Control representation and treatment tensor

The control reproduces the V8.0a scalar feature family. The treatment receives the same observable evidence spatially aligned in a 33×33 tensor. No clean channel, page identity, coordinates or source-group ID enters either estimator.


In [ ]:
def local_mean(a,k=33):return cv2.boxFilter(a.astype(np.float32),-1,(k,k),normalize=True,borderType=cv2.BORDER_REFLECT)
def scalar_images(z):
 o=z['obs'];m=z['mask'];dark=o<190
 inv=(255-o).astype(np.float32)/255
 return {'central_intensity':o.astype(np.float32),'background_normalised_intensity':z['norm'].astype(np.float32),'mask_depth':z['depth'],
 'patch_mean':local_mean(o),'patch_dark_fraction':local_mean(dark),'patch_edge_fraction':local_mean(z['edge']),'patch_mask_fraction':local_mean(m),
 'horizontal_fraction_local':local_mean(z['horizontal']),'vertical_fraction_local':local_mean(z['vertical']),'compact_fraction_local':local_mean(z['compact']),
 'preserve_fraction_local':local_mean(z['preserve']),'remove_fraction_local':local_mean(z['remove']),'ambiguous_fraction_local':local_mean(z['ambiguous']),
 'horizontal_line_response':cv2.boxFilter(inv,-1,(35,1),normalize=True),'vertical_line_response':cv2.boxFilter(inv,-1,(1,15),normalize=True)}
FEATURES=list(scalar_images(next(iter(runtime.values()))).keys())
scalar_cache={p:scalar_images(z) for p,z in runtime.items()}
X_scalar=pd.DataFrame([{f:float(scalar_cache[r.page][f][r.y,r.x]) for f in FEATURES} for r in centres.itertuples()])
CHANNELS=['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge']
def pad_channel(a):return np.pad(a,HALF,mode='reflect')
padded={p:{k:pad_channel((z[k].astype(np.float32)/255 if k in ['obs','norm'] else z[k].astype(np.float32))) for k in CHANNELS} for p,z in runtime.items()}
def patch_tensor(row):return np.stack([padded[row.page][k][row.y:row.y+PATCH,row.x:row.x+PATCH] for k in CHANNELS]).astype(np.float32)
assert not any(x in FEATURES for x in ['page','source_group','x','y','clean','truth'])


## 5. Matched OOF training

Each fold trains both arms on the same training rows and scores the same test rows. The logistic arm is deterministic. The CNN uses one frozen architecture and training schedule; no architecture search is performed.


In [ ]:
class PatchDS(Dataset):
 def __init__(self,indices):self.idx=np.asarray(indices)
 def __len__(self):return len(self.idx)
 def __getitem__(self,i):
  j=int(self.idx[i]);return torch.from_numpy(patch_tensor(centres.iloc[j])),torch.tensor(int(centres.iloc[j].truth),dtype=torch.long)
class TopologyCNN(nn.Module):
 def __init__(self,nc):
  super().__init__();self.net=nn.Sequential(nn.Conv2d(nc,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1));self.head=nn.Linear(64,2)
 def forward(self,x):return self.head(self.net(x).flatten(1))
def train_cnn(tr):
 model=TopologyCNN(len(CHANNELS)).to(DEVICE);opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY);counts=np.bincount(centres.iloc[tr].truth,minlength=2);weights=torch.tensor(counts.sum()/np.maximum(counts,1),dtype=torch.float32,device=DEVICE);lossfn=nn.CrossEntropyLoss(weight=weights)
 loader=DataLoader(PatchDS(tr),batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True)
 model.train()
 for ep in range(EPOCHS):
  total=0
  for xb,yb in loader:
   xb,yb=xb.to(DEVICE,non_blocking=True),yb.to(DEVICE,non_blocking=True);opt.zero_grad(set_to_none=True);loss=lossfn(model(xb),yb);loss.backward();opt.step();total+=float(loss)*len(yb)
  print('epoch',ep+1,'loss',total/len(tr))
 return model
def predict_cnn(model,te):
 model.eval();out=[];loader=DataLoader(PatchDS(te),batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
 with torch.no_grad():
  for xb,_ in loader:out.extend(torch.softmax(model(xb.to(DEVICE)),1)[:,1].cpu().numpy())
 return np.asarray(out)
raw_control=np.full(len(centres),np.nan);raw_topology=np.full(len(centres),np.nan);fold_models=[]
for f in sorted(centres.fold.unique()):
 te=np.where(centres.fold.to_numpy()==f)[0];tr=np.where(centres.fold.to_numpy()!=f)[0]
 control=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(class_weight='balanced',max_iter=3000,random_state=SEED))]);control.fit(X_scalar.iloc[tr],centres.truth.iloc[tr]);raw_control[te]=control.predict_proba(X_scalar.iloc[te])[:,1]
 cnn=train_cnn(tr);raw_topology[te]=predict_cnn(cnn,te);torch.save(cnn.state_dict(),MODELS/f'topology_fold_{f}.pt');fold_models.append({'fold':int(f),'train_rows':len(tr),'test_rows':len(te)})
pd.DataFrame(fold_models).to_csv(AUDIT/'fold_training_audit.csv',index=False)


## 6. Identical cross-fitted calibration and registered decision rule


In [ ]:
def crossfit_cal(raw):
 out=np.full(len(raw),np.nan)
 for f in sorted(centres.fold.unique()):
  te=np.where(centres.fold.to_numpy()==f)[0];tr=np.where(centres.fold.to_numpy()!=f)[0];ok=np.isfinite(raw[tr])
  iso=IsotonicRegression(out_of_bounds='clip').fit(raw[tr][ok],centres.truth.to_numpy()[tr][ok]);out[te]=iso.predict(raw[te])
 return out
prob_control=crossfit_cal(raw_control);prob_topology=crossfit_cal(raw_topology)
def decide(p,pt=PRINT_THRESHOLD):
 p=np.asarray(p);pa=1-p;o=np.full(len(p),'UNCERTAIN',object);o[pa>=ANNOTATION_THRESHOLD]='ANNOTATION';o[(p>=pt)&((p-pa)>=PRINT_MARGIN)]='PRINT';return o
paired=centres.copy();paired['raw_control']=raw_control;paired['p_control']=prob_control;paired['decision_control']=decide(prob_control);paired['raw_topology']=raw_topology;paired['p_topology']=prob_topology;paired['decision_topology']=decide(prob_topology);paired.to_csv(TABLES/'paired_oof_predictions.csv',index=False)


## 7. Full eligible-pixel OOF scoring

Training uses the paired centre sample, but safety evidence is generated by applying each fold-fitted arm to all eligible labelled pixels on the held-out source groups. The notebook reconstructs fold models a second time to keep the implementation direct and auditable.


In [ ]:
# For computational practicality, this starter exports paired centre-level OOF evidence first.
# Full eligible-pixel scoring is mandatory before any freeze. Set RUN_FULL_PIXEL_SCORING=True after the paired audit passes.
RUN_FULL_PIXEL_SCORING=False
if not RUN_FULL_PIXEL_SCORING:
 print('Full-pixel scoring deliberately not run. Paired representation audit only; no freeze is permitted.')


## 8. Paired safe-frontier comparison on OOF centre evidence

This screening comparison cannot substitute for full-pixel safety evidence, but it tests the causal question before the expensive full-page pass.


In [ ]:
def wilson_upper(e,n,z=1.96):
 if n==0:return np.nan
 p=e/n;d=1+z*z/n;c=p+z*z/(2*n);r=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n));return (c+r)/d
def frontier(prob,name):
 rows=[];truth=centres.truth.to_numpy()
 for t in THRESHOLD_GRID:
  sel=(prob>=t)&((2*prob-1)>=PRINT_MARGIN);tp=int(np.sum(sel&(truth==1)));fp=int(np.sum(sel&(truth==0)));n=int(sel.sum())
  rows.append({'arm':name,'threshold':float(t),'selected':n,'true_print':tp,'false_annotation':fp,'false_true_ratio':fp/max(tp,1),'false_preservation_rate':fp/max(n,1),'wilson95_upper':wilson_upper(fp,n),'coverage':n/len(prob),'utility':tp-5*fp,'pages':centres.loc[sel,'page'].nunique(),'source_groups':centres.loc[sel,'source_group'].nunique()})
 return pd.DataFrame(rows)
front=pd.concat([frontier(prob_control,'control_scalar_logistic'),frontier(prob_topology,'treatment_raw_topology_cnn')],ignore_index=True);front.to_csv(TABLES/'paired_frontier_comparison.csv',index=False)
registered=front[np.isclose(front.threshold,PRINT_THRESHOLD)].copy();registered.to_csv(TABLES/'registered_operating_point_comparison.csv',index=False);display(registered)
# Treatment wins/regressions for visual audit.
t=paired.decision_topology=='PRINT';b=paired.decision_control=='PRINT';paired[t&~b].to_csv(TABLES/'topology_only_print.csv',index=False);paired[b&~t].to_csv(TABLES/'control_only_print.csv',index=False);paired[t&b&(paired.truth==0)].to_csv(TABLES/'shared_false_print.csv',index=False)


## 9. Interpretation gate

V8.0b establishes the topology hypothesis only if the treatment improves the paired frontier and then survives the mandatory full-pixel/source-group safety evaluation. It does not freeze or promote either arm.


In [ ]:
manifest_out={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.0b-controlled-topology','development_only':True,'v7_5_reference':str(V75),'development_score_manifest':str(MANIFEST_PATH),'manifest_sha256':sha256_file(MANIFEST_PATH),'fold_map_sha256':json.loads((AUDIT/'fold_map.json').read_text())['sha256'],'pages':len(pages),'source_groups':int(manifest.source_group.nunique()),'paired_rows':len(centres),'patch_size':PATCH,'channels':CHANNELS,'control_features':FEATURES,'control_model':'regularised logistic regression','treatment_model':'fixed compact CNN','same_labels':True,'same_samples':True,'same_folds':True,'same_calibration':True,'same_decision_rule':True,'full_pixel_scoring_completed':RUN_FULL_PIXEL_SCORING,'freeze_permitted':False,'independent_data_used':False,'software':{'python':sys.version,'torch':torch.__version__,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}}
(OUT/'experiment_manifest.json').write_text(json.dumps(manifest_out,indent=2));print(json.dumps(manifest_out,indent=2))
